<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.9em; font-weight:bold; margin:0.6em 0 0.3em 0;">מחברת 3.1: פיצול הנתונים והכנתם לאימון</p>

<p dir="rtl" style="direction:rtl; text-align:right;">זהו המשך מודרך של חלק א במפגש 3. השאלה: האם אפשר לנבא איך תיראה היד של מטופל אחרי שנה? יש לנו 49 מטופלים אמיתיים אחרי שבץ. לפני שמאמנים מודל צריך לעשות שני דברים: <b>לנעול</b> חלק מהמטופלים בצד (קבוצת הבדיקה), ו<b>להכין</b> את הנתונים כך שכל מספר שהמודל "לומד" יבוא מקבוצת האימון בלבד. נעשה זאת קודם ביד, בעזרת <code dir="ltr">dplyr</code>, ואחר כך בעזרת מתכון (<code dir="ltr">recipe</code>), המקבילה ב-<code dir="ltr">tidymodels</code>. משך מוערך: כ-15 דקות. בכל תא יש פקודה אחת שמדפיסה תוצאה אחת.</p>

<p dir="rtl" style="direction:rtl; text-align:right;"><b>כך עובד כל תא בדיקה:</b> אם התשובה נכונה יודפס <code dir="ltr">OK</code>. אם לא, תופיע שגיאה, וזה בסדר: תקנו והריצו שוב. כל מקום שבו כתוב <code dir="ltr">___</code> מחכה להשלמה שלכם.</p>

<p dir="rtl" style="direction:rtl; text-align:right;"><b>תזכורת: <span dir="ltr">FMA-UE</span>.</b> <span dir="ltr">Fugl-Meyer</span> לגף העליון מודד תפקוד מוטורי אחרי שבץ. מבקשים תנועות לפי פרוטוקול, וכל פריט מקבל 0, 1 או 2. הטווח הוא 0 עד 66, וגבוה = טוב יותר. קריאה נוספת: <span dir="ltr"><a href="https://www.sralab.org/rehabilitation-measures/fugl-meyer-assessment-motor-recovery-after-stroke">Shirley Ryan AbilityLab</a></span>, <span dir="ltr"><a href="https://pubmed.ncbi.nlm.nih.gov/1135616/">PubMed 1135616</a></span>.</p>

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">הכנת הסביבה</p>

<p dir="rtl" style="direction:rtl; text-align:right;">התא הבא מתקין רק את החבילות הדרושות (<code dir="ltr">tidymodels</code> ו-<code dir="ltr">kknn</code>) ממאגר בינארי מהיר, ומדפיס כמה זמן ההתקנה נמשכה. הריצו אותו פעם אחת בתחילת המפגש.</p>

In [ ]:
# Install only the packages the course needs, from a binary repository (fast)
options(repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/jammy/latest"))
options(HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
        paste(getRversion(), R.version$platform, R.version$arch, R.version$os)))
# tidymodels + kknn: about 100 packages in all, 43 more than tidyverse alone. From the Posit binary
# repository there is no compiling, but the time on a fresh Colab session was NOT measured here
# (a rough guess is a few minutes). Run this cell first, before class starts.
needed  <- c("tidymodels", "kknn")   # add packages here when a notebook needs them
missing <- setdiff(needed, rownames(installed.packages()))   # skip installed ones
t <- system.time(if (length(missing) > 0) install.packages(missing))
cat("Installed:", if (length(missing)) paste(missing, collapse = ", ") else "nothing new",
    "| elapsed seconds:", round(t[["elapsed"]], 1), "\n")
suppressPackageStartupMessages(library(tidymodels))   # recipes, parsnip, rsample, yardstick, tune, ggplot2, dplyr ...
tidymodels_prefer()                                   # when two packages share a name, prefer the tidymodels one

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">טעינת הנתונים</p>

<p dir="rtl" style="direction:rtl; text-align:right;">הפונקציה <code dir="ltr">load_course_data()</code> מורידה את הטבלה הנקייה של 49 המטופלים (סוף פרק 5). הטבלה <code dir="ltr">stroke_ml</code> היא טבלת הלמידה של פרק 8: המטרה היא הציון הכרוני <code dir="ltr">fma_sum_chr</code> (וגרסה קטגורית שלו, <code dir="ltr">outcome_mild</code>: ציון 43 ומעלה = ליקוי קל), והמאפיינים הם מה שידוע כבר בשלב התת-חריף.</p>

In [ ]:
source("https://raw.githubusercontent.com/ShayOfir/DSAI4PT_Zfat_Students/main/R/load_course_data.R")

In [ ]:
stroke <- load_course_data("stroke_clean") |>        # the cleaned table: one row per patient
  as_tibble() |>                                      # make it a tibble
  mutate(sex = factor(sex, levels = c("F", "M")),     # categories are stored as factors
         lesion_type = factor(lesion_type, levels = c("ischemic", "hemorrhagic", "mixed")))

In [ ]:
stroke_ml <- stroke |>
  select(id, fma_sum_chr,                                   # id and the chronic score (the target)
         fma_sum_sub, fma_a_sub, fma_bc_sub,                # features known at the subacute visit ...
         age, sex, lesion_type, lesion_vol, tao_sub) |>     # ... as in chapter 8 of the book
  mutate(outcome_mild = factor(if_else(fma_sum_chr >= 43, "mild", "not_mild"),   # the categorical target
                               levels = c("mild", "not_mild")))

In [ ]:
glimpse(stroke_ml)   # one line per column: name, type, first values

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">משימה 1: מכירים את הטבלה</p>

<p dir="rtl" style="direction:rtl; text-align:right;">ערך לוגי <code dir="ltr">TRUE</code> נספר כ-1, ולכן <code dir="ltr">mean()</code> של ערכים לוגיים הוא <b>שיעור</b>. דוגמה: שיעור המטופלים שגילם מעל 60.</p>

In [ ]:
mean(stroke_ml$age > 60)   # share of patients older than 60

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף א.</b> כמה ערכים חסרים יש בעמודה <code dir="ltr">sex</code>? הפונקציה <code dir="ltr">is.na()</code> בודקת כל ערך, ו-<code dir="ltr">sum()</code> סופרת את ה-<code dir="ltr">TRUE</code>.</p>

In [ ]:
n_missing_sex <- sum(is.na(stroke_ml$___))   # TODO: which column?

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(n_missing_sex == 1)
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף ב.</b> מה שיעור המטופלים ש"יגיעו לליקוי קל" (<code dir="ltr">outcome_mild == "mild"</code>)? השתמשו באותו רעיון כמו בדוגמה.</p>

In [ ]:
share_mild <- mean(stroke_ml$outcome_mild == ___)   # TODO: which category counts?

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(abs(share_mild - 34/49) < 1e-9)
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">משימה 2: מפצלים את הנתונים</p>

<p dir="rtl" style="direction:rtl; text-align:right;"><b>הכלל:</b> קודם מפצלים, ורק אחר כך עושים כל דבר אחר. קבוצת הבדיקה היא "מגירה נעולה": לא מסתכלים בה, לא מחשבים ממנה ממוצעים ולא בוחרים לפיה מודל, עד לסוף. <code dir="ltr">initial_split()</code> מפצלת באקראי. הארגומנט <code dir="ltr">prop</code> הוא חלק האימון, ו-<code dir="ltr">set.seed()</code> קובעת את האקראיות כך שכולם מקבלים אותו פיצול. דוגמה לפיצול חצי-חצי:</p>

In [ ]:
set.seed(1)                                            # fixed seed
example_split <- initial_split(stroke_ml, prop = 0.5)   # half training, half test
example_split                                          # <training / testing / total>

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף א.</b> בצעו פיצול של 75% אימון ו-25% בדיקה, והוסיפו <code dir="ltr">strata = outcome_mild</code>: הארגומנט הזה שומר שבשתי הקבוצות יהיה שיעור דומה של "ליקוי קל" (פיצול מרובד). שמרו ב-<code dir="ltr">stroke_split</code>.</p>

In [ ]:
set.seed(820)                                                 # fixed seed
stroke_split <- initial_split(stroke_ml, prop = ___, strata = ___)   # TODO: the share for training, and the column to stratify by

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(nrow(training(stroke_split)) == 36, nrow(testing(stroke_split)) == 13)
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף ב.</b> הפונקציות <code dir="ltr">training()</code> ו-<code dir="ltr">testing()</code> שולפות את שתי הטבלאות מהפיצול. שמרו אותן ב-<code dir="ltr">stroke_train</code> וב-<code dir="ltr">stroke_test</code>.</p>

In [ ]:
stroke_train <- training(___)   # TODO: the training table
stroke_test  <- testing(___)    # TODO: the test table

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(nrow(stroke_train) == 36, nrow(stroke_test) == 13)
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף ג.</b> האם הריבוד עבד? דוגמה: כך סופרים את הקטגוריות באימון.</p>

In [ ]:
count(stroke_train, outcome_mild)   # patients per category, training table

<p dir="rtl" style="direction:rtl; text-align:right;">חשבו את שיעור "ליקוי קל" בקבוצת האימון וגם בקבוצת הבדיקה (כמו בסעיף ב של משימה 1).</p>

In [ ]:
share_mild_train <- mean(stroke_train$outcome_mild == "mild")   # training share
share_mild_test  <- mean(stroke_test$___ == "mild")             # TODO: the test share

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(abs(share_mild_train - share_mild_test) < 0.05)   # stratified: nearly equal
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;"><b>תובנה.</b> מדוע חשוב ששיעור "ליקוי קל" יהיה כמעט זהה בקבוצת האימון ובקבוצת הבדיקה?</p>
<p dir="rtl" style="direction:rtl; text-align:right;">כתבו כאן משפט או שניים: ...</p>

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">משימה 3: מכינים את הנתונים, ולומדים מהאימון בלבד</p>

<p dir="rtl" style="direction:rtl; text-align:right;">שלוש פעולות הכנה: <b>השלמת ערך חסר</b>, <b>תקנון</b> (z-score: מחסירים ממוצע ומחלקים בסטיית תקן, כדי שגיל, ציון ונפח נגע יהיו באותה סקאלה, וזה חשוב ל-K-NN), ו<b>משתני דמה</b> (קטגוריה הופכת לעמודה 0/1). הכלל לכולן: <b>המספרים נלמדים מטבלת האימון בלבד, ואז מופעלים על האימון ועל הבדיקה.</b> נעבוד עם ארבעה מאפיינים: <code dir="ltr">fma_sum_sub</code>, <code dir="ltr">age</code>, <code dir="ltr">sex</code> ו-<code dir="ltr">lesion_vol</code>.</p>

<p dir="rtl" style="direction:rtl; text-align:right;"><b>3.1 השלמת ערך חסר.</b> חסר ערך אחד של <code dir="ltr">sex</code>, והוא נמצא בקבוצת האימון. בקטגוריה משלימים בערך הנפוץ ביותר (השכיח) <b>באימון</b>. דוגמה: טבלת השכיחות באימון.</p>

In [ ]:
table(stroke_train$sex)   # counts per category in the training table (missing values are not counted)

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף א.</b> הערך הנפוץ הוא "M" או "F"? שמרו אותו כגורם (<code dir="ltr">factor</code>) ב-<code dir="ltr">sex_mode</code>.</p>

In [ ]:
sex_mode <- factor("___", levels = c("F", "M"))   # TODO: "F" or "M"?

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(as.character(sex_mode) == names(which.max(table(stroke_train$sex))))
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;">הפונקציה <code dir="ltr">replace_na(x, value)</code> מחליפה כל <code dir="ltr">NA</code> ב-<code dir="ltr">x</code> בערך <code dir="ltr">value</code>. דוגמה: השלמה בטבלת האימון (נשמרת ב-<code dir="ltr">stroke_train_f</code>).</p>

In [ ]:
stroke_train_f <- stroke_train |>
  mutate(sex = replace_na(sex, sex_mode))   # fill the missing sex with the training mode
sum(is.na(stroke_train_f$sex))              # how many gaps are left in training?

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף ב.</b> בקבוצת הבדיקה הזאת אין חוסרים, אבל מטופל חדש בקליניקה עשוי להגיע עם נתון חסר. לכן מפעילים את אותו כלל גם על הבדיקה, <b>עם אותו</b> <code dir="ltr">sex_mode</code> (לא מחשבים שכיח חדש מהבדיקה!). שמרו ב-<code dir="ltr">stroke_test_f</code>.</p>

In [ ]:
stroke_test_f <- stroke_test |>
  mutate(sex = replace_na(sex, ___))   # TODO: which value?

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(sum(is.na(stroke_test_f$sex)) == 0, sum(is.na(stroke_train_f$sex)) == 0)
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;"><b>3.2 תקנון.</b> דוגמה: הממוצע וסטיית התקן של הגיל, מחושבים <b>מהאימון בלבד</b>, ואז מופעלים על שתי הטבלאות.</p>

In [ ]:
age_mean <- mean(stroke_train_f$age)   # learned from training
age_sd   <- sd(stroke_train_f$age)     # learned from training
c(age_mean = age_mean, age_sd = age_sd)

In [ ]:
stroke_train_z <- stroke_train_f |> mutate(age = (age - age_mean) / age_sd)   # training table: z-score of age
stroke_test_z  <- stroke_test_f  |> mutate(age = (age - age_mean) / age_sd)   # test table: the SAME training numbers
round(c(train_mean = mean(stroke_train_z$age), train_sd = sd(stroke_train_z$age)), 2)

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף א.</b> נפח הנגע מוטה מאוד (כמה נגעים גדולים מאוד), ולכן לפני התקנון לוקחים לוגריתם (<code dir="ltr">log()</code>). ללוגריתם אין מספרים שנלמדים, ולכן מפעילים אותו על שתי הטבלאות כמו שהוא. עשו זאת, ושמרו בעמודה <code dir="ltr">lesion_vol</code> עצמה.</p>

In [ ]:
stroke_train_z <- stroke_train_z |> mutate(lesion_vol = ___(lesion_vol))   # TODO: the log function, training
stroke_test_z  <- stroke_test_z  |> mutate(lesion_vol = ___(lesion_vol))   # TODO: the same function, test
summary(stroke_train_z$lesion_vol)

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(all(abs(stroke_train_z$lesion_vol - log(stroke_train$lesion_vol)) < 1e-9), all(abs(stroke_test_z$lesion_vol - log(stroke_test$lesion_vol)) < 1e-9))
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף ב.</b> עכשיו מתקננים את הלוגריתם של נפח הנגע. חשבו ממוצע וסטיית תקן <b>מטבלת האימון</b>.</p>

In [ ]:
vol_mean <- mean(stroke_train_z$___)   # TODO: learned from training
vol_sd   <- sd(stroke_train_z$___)     # TODO: learned from training
c(vol_mean = vol_mean, vol_sd = vol_sd)

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(abs(vol_mean - mean(log(stroke_train$lesion_vol))) < 1e-9, abs(vol_sd - sd(log(stroke_train$lesion_vol))) < 1e-9)
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף ג.</b> הפעילו את המספרים על שתי הטבלאות. שימו לב: גם בבדיקה משתמשים ב-<code dir="ltr">vol_mean</code> וב-<code dir="ltr">vol_sd</code> של <b>האימון</b>.</p>

In [ ]:
stroke_train_z <- stroke_train_z |> mutate(lesion_vol = (lesion_vol - ___) / ___)   # TODO: the learned numbers, training
stroke_test_z  <- stroke_test_z  |> mutate(lesion_vol = (lesion_vol - ___) / ___)   # TODO: the same numbers, test
round(c(train_mean = mean(stroke_train_z$lesion_vol), train_sd = sd(stroke_train_z$lesion_vol)), 2)

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(abs(mean(stroke_train_z$lesion_vol)) < 1e-9, abs(sd(stroke_train_z$lesion_vol) - 1) < 1e-9, abs(mean(stroke_test_z$lesion_vol)) > 1e-6)
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף ד.</b> נשאר הציון התת-חריף <code dir="ltr">fma_sum_sub</code>. תקננו אותו בעצמכם, בשלושה תאים: ממוצע, סטיית תקן, והפעלה על שתי הטבלאות (כמו <code dir="ltr">age</code> בדוגמה).</p>

In [ ]:
fma_mean <- mean(stroke_train_f$___)   # TODO: learned from training
fma_sd   <- sd(stroke_train_f$___)     # TODO: learned from training

In [ ]:
stroke_train_z <- stroke_train_z |> mutate(fma_sum_sub = (fma_sum_sub - ___) / ___)   # TODO: training
stroke_test_z  <- stroke_test_z  |> mutate(fma_sum_sub = (fma_sum_sub - ___) / ___)   # TODO: test, the same numbers

In [ ]:
# Visible check: stops with an error if the answer is wrong
num_cols <- c("age", "lesion_vol", "fma_sum_sub")
stopifnot(all(abs(colMeans(stroke_train_z[num_cols])) < 1e-9),                 # training: mean 0
          all(abs(sapply(stroke_train_z[num_cols], sd) - 1) < 1e-9),           # training: sd 1
          all(abs(colMeans(stroke_test_z[num_cols])) > 1e-6))                  # test: NOT exactly 0
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;">ממוצע הגיל בטבלת הבדיקה אחרי התקנון (שימו לב שהוא לא בדיוק 0):</p>

In [ ]:
mean(stroke_test_z$age)   # test mean after scaling with the TRAINING numbers

<p dir="rtl" style="direction:rtl; text-align:right;"><b>תובנה.</b> בטבלת האימון הממוצע אחרי תקנון הוא בדיוק 0, ובבדיקה לא. האם זו טעות? מה היה קורה אילו חישבנו את הממוצע מכל 49 המטופלים?</p>
<p dir="rtl" style="direction:rtl; text-align:right;">כתבו כאן משפט או שניים: ...</p>

<p dir="rtl" style="direction:rtl; text-align:right;"><b>3.3 משתנה דמה.</b> דוגמה: הקטגוריה <code dir="ltr">sex</code> הופכת לעמודה <code dir="ltr">sex_M</code>: 1 לגברים ו-0 לנשים (<code dir="ltr">F</code> היא קבוצת הייחוס, ולכן אין לה עמודה). <code dir="ltr">if_else(תנאי, אם נכון, אם לא)</code> בוחרת ערך לכל שורה.</p>

In [ ]:
stroke_train_z |>
  mutate(sex_M = if_else(sex == "M", 1, 0)) |>   # 1 = male, 0 = female
  select(sex, sex_M) |>
  head(4)

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף א.</b> בנו את טבלת האימון המוכנה <code dir="ltr">stroke_train_d</code>: הוסיפו את <code dir="ltr">sex_M</code>, ו-<code dir="ltr">select()</code> משאירה רק את המאפיינים ואת המטרה.</p>

In [ ]:
stroke_train_d <- stroke_train_z |>
  mutate(sex_M = if_else(sex == "___", 1, 0)) |>                # TODO: which category gets 1?
  select(fma_sum_sub, age, lesion_vol, sex_M, outcome_mild)     # features, then the target
glimpse(stroke_train_d)

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף ב.</b> עכשיו אותו הדבר בטבלת הבדיקה, <code dir="ltr">stroke_test_d</code> (מתחילים מ-<code dir="ltr">stroke_test_z</code>), בלי רמזים.</p>

In [ ]:
# TODO: build stroke_test_d exactly like stroke_train_d, starting from stroke_test_z
stroke_test_d <- ___

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(ncol(stroke_train_d) == 5, ncol(stroke_test_d) == 5, nrow(stroke_test_d) == 13, !anyNA(stroke_test_d), all(stroke_test_d$sex_M %in% c(0, 1)))
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">משימה 4: המקבילה ב-tidymodels, מתכון (recipe)</p>

<p dir="rtl" style="direction:rtl; text-align:right;">את כל מה שעשינו ביד אפשר לכתוב כ<b>מתכון</b>: תוכנית כתובה של שלבי ההכנה. <code dir="ltr">recipe()</code> מקבלת נוסחה (מטרה <code dir="ltr">~</code> מאפיינים) ואת טבלת האימון. כל <code dir="ltr">step_*()</code> מוסיפה שלב. עד כאן לא חושב דבר. <code dir="ltr">prep()</code> <b>לומדת</b> את המספרים מטבלת האימון, ו-<code dir="ltr">bake()</code> <b>מפעילה</b> אותם על כל טבלה. הבוררים <code dir="ltr">all_numeric_predictors()</code> ו-<code dir="ltr">all_nominal_predictors()</code> פירושם "כל המנבאים המספריים" ו"כל המנבאים הקטגוריאליים".</p>

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף א.</b> ארבעה שלבים, בסדר שעשינו ביד: השלמה, לוגריתם, תקנון, משתנה דמה. שלושת הראשונים נתונים. הוסיפו את הרביעי (הסדר חשוב: דמה אחרי התקנון, כדי שעמודת 0/1 לא תתוקנן).</p>

In [ ]:
stroke_rec <- recipe(outcome_mild ~ fma_sum_sub + age + sex + lesion_vol,
                     data = stroke_train) |>      # the plan: which columns, and which is the target
  step_impute_mode(sex) |>                         # given: fill a missing category with the most common one
  step_log(lesion_vol) |>                          # given: log of the lesion volume
  step_normalize(all_numeric_predictors()) |>      # given: z-score for every numeric predictor
  ___                                              # TODO: add the step that turns categories into 0/1 columns

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(length(stroke_rec$steps) == 4)
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף ב.</b> <code dir="ltr">prep()</code> לומדת מהאימון. <code dir="ltr">tidy(..., number = 1)</code> מראה מה נלמד בשלב הראשון (השכיח של <code dir="ltr">sex</code>).</p>

In [ ]:
stroke_rec_prep <- prep(stroke_rec)   # learn the numbers from the training table
tidy(stroke_rec_prep, number = 1)        # what step 1 (impute mode) learned

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף ג.</b> <code dir="ltr">bake()</code> מפעילה את המתכון על טבלה (<code dir="ltr">new_data</code>). הפעלה על האימון נתונה. הפעילו אותו על טבלת הבדיקה.</p>

In [ ]:
train_baked <- bake(stroke_rec_prep, new_data = stroke_train)   # given
test_baked  <- bake(stroke_rec_prep, new_data = ___)            # TODO: the table to apply the recipe to
glimpse(test_baked)

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(nrow(test_baked) == 13, nrow(train_baked) == 36, !anyNA(test_baked))
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">משימה 5: בודקים שקבוצת הבדיקה לא נגעה באימון</p>

<p dir="rtl" style="direction:rtl; text-align:right;">שלוש בדיקות. הראשונה: אף מטופל לא נמצא בשתי הקבוצות. הפונקציה <code dir="ltr">intersect(a, b)</code> מחזירה את הערכים המשותפים. בדוגמה, רק 3 משותף, ולכן התוצאה 1.</p>

In [ ]:
length(intersect(c(1, 2, 3), c(3, 4)))   # how many values are in both vectors?

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף א.</b> כמה מזהי מטופלים (<code dir="ltr">id</code>) משותפים לאימון ולבדיקה?</p>

In [ ]:
n_overlap <- length(intersect(stroke_train$___, stroke_test$___))   # TODO: the id column of both tables

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(n_overlap == 0, nrow(stroke_train) + nrow(stroke_test) == nrow(stroke_ml))
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;">השנייה: המספרים שנלמדו הם מהאימון. הממוצע של הגיל באימון (שנלמד) מול הממוצע בכל 49 המטופלים (שהיינו "מזהמים" בבדיקה):</p>

In [ ]:
c(training_only = age_mean, all_49_patients = mean(stroke_ml$age))   # a leak would use the second one

<p dir="rtl" style="direction:rtl; text-align:right;">השלישית והחשובה: הכתיבה הידנית והמתכון נותנים <b>בדיוק אותן תוצאות</b>. <code dir="ltr">all.equal(a, b)</code> מחזירה <code dir="ltr">TRUE</code> כששני וקטורים שווים. השוו את עמודת הגיל בטבלת הבדיקה:</p>

In [ ]:
all.equal(test_baked$age, stroke_test_d$age)   # recipe vs manual: age (z-score)

<p dir="rtl" style="direction:rtl; text-align:right;">ועכשיו עמודת הדמה <code dir="ltr">sex_M</code>:</p>

In [ ]:
all.equal(test_baked$___, stroke_test_d$___)   # TODO: compare the dummy column sex_M in both tables

In [ ]:
# Visible check: stops with an error if the answer is wrong
num_cols <- c("fma_sum_sub", "age", "lesion_vol", "sex_M")
stopifnot(isTRUE(all.equal(as.data.frame(test_baked[num_cols]), as.data.frame(stroke_test_d[num_cols]), check.attributes = FALSE)),
          isTRUE(all.equal(as.data.frame(train_baked[num_cols]), as.data.frame(stroke_train_d[num_cols]), check.attributes = FALSE)))
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;"><b>תובנה.</b> למה עדיף לעבוד עם מתכון (<code dir="ltr">recipe</code>) ולא עם הכתיבה הידנית, כשעובדים עם אימות צולב (בהמשך המפגש)?</p>
<p dir="rtl" style="direction:rtl; text-align:right;">כתבו כאן משפט או שניים: ...</p>